## <a href="https://cursos.alura.com.br/course/langchain-python-ferramentas-llm-openai/task/156165"><b>Utilizando LCEL para criar um roteiro de viagens</b></a><br/>

In [1]:
from langchain.prompts import ChatPromptTemplate # Uma interação de múltiplos prompts é um chat, por isso, vamos importar um ChatPromptTemplate

In [2]:
%pip install -qr requirements.txt

Note: you may need to restart the kernel to use updated packages.


#### <b>PASSO 1 - IMPORTS e CRIAÇÃO DA LLM</b>

In [3]:
from langchain_openai import ChatOpenAI
from os import getenv
from dotenv import load_dotenv # CARREGA A VARIÁVEL DE AMBIENTE OPENAI_KEY LIDA DO ARQUIVO .env

load_dotenv() # CARREGANDO O ARQUIVO COM A OPENAI_KEY

llm = ChatOpenAI( # INSTANCIANDO A LLM
                    model="gpt-4.1-mini",
                    temperature=0.5,
                    # 1 - OBTENDO A API KEY POR MEIO DA VARIÁVEL DE AMBIENTE OPENAI_KEY. QUE VAI FICAR ARMAZENADA NO ARQUIVO .env.
                    # 2 - AINDA É NECESSÁRIO CARREGAR ESSE ARQUIVO. VER NA PRIMEIRA CÉLULA DO NOTEBOOK
                    api_key=getenv("OPENAI_KEY")                    
                )

#### <b>PASSO 2 - CRIANDO O <i>PROMPT TEMPLATE</i> E ASSOCIANDO UM PARSER A ELE</b></br> 

<b><ol><li>CRIANDO OS MODELOS</li></ol></b>

<ul><ul><li><b>CRIANDO O PARSER E ASSOCIANDO ELE AO MODELO DE CIDADE</b></li></ul></ul>

In [4]:
from pydantic import Field,BaseModel # pydantic -> Biblioteca para validação de dados. Garante que os dados recebidos ou manipulados estejam no formato correto,
                                     # BaseModel -> Os modelos pydantic são classes que herdam BaseModel (https://docs-pydantic-dev.translate.goog/latest/concepts/models/?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc)
                                     # Modelos possuem campos como atributos.
                                     
                                     # Field -> Para personalizar os campos do modelo (https://docs-pydantic-dev.translate.goog/latest/concepts/fields/?_x_tr_sl=en&_x_tr_tl=pt&_x_tr_hl=pt&_x_tr_pto=tc)

class Destino(BaseModel): # A nossa classe vai estender a classe BaseModel, que terá dois campos, a cidade e o motivo de visitá-la
    cidade: str = Field(description="cidade a visitar") # Descrição do campo. Apenas informativo
    motivo: str = Field(description="motivo pelo qual é interessante visitar") # Descrição do campo. Apenas informativo  

from langchain import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser # EXISTEM DIVERSOS OUTPUT PARSERS (https://python.langchain.com/docs/concepts/output_parsers/)

parseador = JsonOutputParser(pydantic_object=Destino) # DOCUMENTAÇÃO JsonOutputParser (https://python.langchain.com/docs/how_to/output_parser_json)

template_cidade = PromptTemplate(
                                    template="""Sugira uma cidade, dado o meu interesse por {interesse}.
                                    {formatacao_de_saida_da_ia}""", # AQUI TEMOS UMA VARÍAVEL PARCIAL. UTLIZAÇÃO DA TÉCNICA DE SHOTS PARA PROMPTS
                                    input_variables=["interesse"],
                                    # A VARÍAVEL PARCIAL É UM DICIONÁRIO. FUNCIONA COMO O SHOT
                                    partial_variables={"formatacao_de_saida_da_ia":parseador.get_format_instructions()}, # PASSANDO O PARSEADOR PARA A VARIÁVEL FORMATAÇÃO DE SAÍDA.
                                                                                                   
                                ) # INSTANCIANDO PromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

<ul><ul><li><b>CRIANDO O MODELO PARA RESTAURANTES</b></li></ul></ul>

In [5]:
template_restaurante = ChatPromptTemplate.from_template("Sugira restaurantes populares entre locais na {cidade}") # INSTANCIANDO ChatPromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

<ul><ul><li><b>CRIANDO O MODELO CULTURAL</b></li></ul></ul>

In [6]:
template_cultural = ChatPromptTemplate.from_template("Sugira atividades e locais culturais em {cidade}") # INSTANCIANDO ChatPromptTemplate e INICIANDO A PARTIR DE UM TEMPLATE

#### <b>PASSO 3 - USANDO A <a ref="https://python.langchain.com/docs/concepts/lcel/#composition-syntax">LCEL</a></b>
<ul><li>O resultado de um vai jogando no outro</li></ul>

In [ ]:
chain1 = template_cidade |llm |parseador # O PARSEADOR É UM JSON, DICIONÁRIO, TRANSFORMADO EM OBJETO PYTHON

In [8]:
# FAZENDO UMA CHAMADA PARA O TEMPLATE
resposta = chain1.invoke({"interesse":"praias"}) # AQUI ESTAMOS CHAMANDO A PRIMEIRA CHAIN, PASSANDO O INTERESSE COMO ENTRADA
print(resposta)

{'cidade': 'Florianópolis', 'motivo': 'Florianópolis é famosa por suas belas praias, com opções para todos os gostos, desde praias calmas para relaxar até praias com ótimas condições para surf.'}


<ul><ul><ul><b>Jogando a cidade da cadeia 1 para a cadeia 2, com parseador de string StrOutputParser</b></ul></ul></ul>

<font style="color:lightgreen">[llm/start]</font> [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [</ul>
   <ul><ul>"Human: Sugira uma cidade, dado o meu interesse por praias.\n                                    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{\"properties\": {\"cidade\": {\"description\": \"cidade a visitar\", \"title\": \"Cidade\", \"type\": \"string\"}, \"motivo\": {\"description\": \"motivo pelo qual é interessante visitar\", \"title\": \"Motivo\", \"type\": \"string\"}}, \"required\": [\"cidade\", \"motivo\"]}\n```"</ul></ul>
  <ul>]</ul>
}<br/>
<font style="color:lightblue">[llm/end]</font> [chain:RunnableSequence > llm:ChatOpenAI] [1.53s] Exiting LLM run with output:<br/>
{
  <ul>"text": "{\n  \"cidade\": \"Florianópolis\",\n  \"motivo\": \"Florianópolis é conhecida por suas belas praias, com opções para todos os gostos, desde praias calmas para relaxar até praias com boas ondas para surf.\"\n}"</ul>
}

<font style="color:lightgreen">[llm/start]</font>[chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
<ul> "prompts": [</ul>
    <ul><ul>"Human: Sugira restaurantes populares entre locais na Florianópolis"</ul></ul>
  ]
}<br/>
<font style="color:lightblue">[llm/end]</font> [chain:RunnableSequence > llm:ChatOpenAI] [8.17s] Exiting LLM run with output:<br/>
{
  <ul>"text": "Claro! Florianópolis é conhecida por sua gastronomia diversificada, especialmente frutos do mar frescos e pratos típicos da culinária catarinense. Aqui estão alguns restaurantes populares entre os locais que você pode gostar:\n\n1. **Ostradamus** – Localizado na Lagoa da Conceição, é famoso por suas ostras frescas e pratos à base de frutos do mar. Ambiente descontraído e ótimo para quem quer experimentar a culinária local.\n\n2. **Bar do Arante** – Um clássico em Florianópolis, na Praia do Matadeiro. Conhecido pelo peixe na telha e ambiente simples, é muito frequentado por moradores e turistas que buscam comida caseira e saborosa.\n\n3. **Restaurante do Chico** – Também na Lagoa da Conceição, oferece pratos típicos da região, como sequência de camarão e mariscos, em um ambiente rústico e acolhedor.\n\n4. **Box 32** – Localizado no Mercado Público de Florianópolis, é um ponto tradicional para comer frutos do mar frescos, como a famosa sequência de camarão. Muito frequentado por locais para um almoço rápido e saboroso.\n\n5. **Canto dos Açores** – Restaurante que valoriza a culinária açoriana, muito presente na cultura local. Fica no bairro Ribeirão da Ilha, onde também é possível encontrar ostras cultivadas na região.\n\n6. **Ponto G** – Em Santo Antônio de Lisboa, é conhecido pela comida regional com um toque contemporâneo, além da vista charmosa para o mar.\n\nSe quiser, posso ajudar com sugestões específicas de pratos ou bairros para focar!"</ul>
}

In [9]:
from langchain_core.output_parsers import StrOutputParser
#from langchain.globals import set_debug
#set_debug(True)

chain2 = template_restaurante | llm |StrOutputParser() # A saída do llm é uma string, por isso, utilizamos o StrOutputParser

chain = chain1 | chain2 # PARA PODER PASSAR A CIDADE PARA O RESTAURANTE, PRECISAMOS CRIAR UMA NOVA CHAIN QUE RECEBA A PRIMEIRA E A SEGUNDA
resposta = chain.invoke({"interesse":"praias"}) 
print(resposta) 

Claro! Florianópolis é conhecida por sua culinária deliciosa, especialmente frutos do mar frescos. Aqui estão alguns restaurantes populares entre os moradores locais:

1. **Ostradamus** – Localizado na Lagoa da Conceição, é famoso pelas ostras frescas e pratos com frutos do mar. Ambiente descontraído e ótimo para experimentar a culinária típica da região.

2. **Bar do Arante** – Um clássico na Praia do Moçambique, conhecido pelo peixe frito e ambiente rústico à beira-mar. Muito frequentado por locais que apreciam comida simples e saborosa.

3. **Box 32** – Na tradicional Feira do Largo da Alfândega, no centro, é uma ótima opção para experimentar pratos típicos catarinenses, como sequência de camarão e tainha.

4. **Restaurante Ponta das Caranhas** – Situado na Praia dos Ingleses, oferece pratos variados com frutos do mar e uma vista incrível para o mar.

5. **Cantina Giardino** – Localizada no bairro Santa Mônica, é uma cantina italiana muito apreciada pelos locais, com massas caseiras

<ul><ul><ul><b>Sem o parseador de String</b></ul></ul></ul>


In [10]:
chain2 = template_restaurante | llm # A saída do llm é uma string, por isso, utilizamos o StrOutputParser
chain = chain1 | chain2 # PARA PODER PASSAR A CIDADE PARA O RESTAURANTE, PRECISAMOS CRIAR UMA NOVA CHAIN QUE RECEBA A PRIMEIRA E A SEGUNDA
resposta = chain.invoke({"interesse":"praias"}) 
print(resposta) 

content='Claro! Florianópolis é famosa pela sua culinária deliciosa, especialmente frutos do mar, e tem muitos restaurantes populares entre os locais. Aqui vão algumas sugestões:\n\n1. **Ostradamus** – Localizado na Lagoa da Conceição, é conhecido pelas ostras frescas e pratos à base de frutos do mar. Ambiente aconchegante e ótimo para experimentar especialidades locais.\n\n2. **Bar do Arante** – Na Praia do Pântano do Sul, é um clássico para quem quer comer peixe fresco e frutos do mar em um ambiente simples e tradicional.\n\n3. **Restaurante Rancho Açoriano** – Também no Pântano do Sul, famoso pela sequência de camarão e pratos típicos da culinária açoriana.\n\n4. **Box 32** – Localizado no Mercado Público de Florianópolis, é um dos pontos preferidos para quem quer provar pratos tradicionais catarinenses, como a sequência de camarão.\n\n5. **Cantina da Lua** – Na Lagoa da Conceição, oferece comida italiana e frutos do mar, com um ambiente descontraído e música ao vivo.\n\n6. **Restau

<ul><ul><ul><b>Executando até a cadeia 2 com debug ativado, para verificar se na cadeia 2, a cidade é fornecida como dicionário</b></ul></ul></ul>
<ul><ul><ul><ul>A conclusão é que não</ul></ul></ul></ul>

In [11]:
from langchain_core.output_parsers import StrOutputParser
from langchain.globals import set_debug
set_debug(True)

chain2 = template_restaurante | llm |StrOutputParser() # A saída do llm é uma string, por isso, utilizamos o StrOutputParser

chain = chain1 | chain2 # PARA PODER PASSAR A CIDADE PARA O RESTAURANTE, PRECISAMOS CRIAR UMA NOVA CHAIN QUE RECEBA A PRIMEIRA E A SEGUNDA
resposta = chain.invoke({"interesse":"praias"}) 
print(resposta) 

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "interesse": "praias"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "interesse": "praias"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Sugira uma cidade, dado o meu interesse por praias.\n                                    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is

#### <b>PASSO 4 - INVOCANDO A CADEIA GERAL</b>

<ul><ul><ul><b>Jogando da cadeia 2 para a cadeia 3, sendo a saída da cadeia 2 uma string</b></ul></ul></ul>

<font style="color:lightgreen">[llm/start]</font>[chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [
    "Human: Sugira atividades e locais culturais em <b>Claro! Florianópolis é conhecida por sua culinária diversificada, especialmente frutos do mar frescos. Aqui estão alguns restaurantes populares entre os locais que você pode gostar de visitar:\n\n1. **Ostradamus**  \n   Localização: Ribeirão da Ilha  \n   Destaque: Especializado em ostras e frutos do mar frescos, é um dos restaurantes mais tradicionais da ilha. Ambiente agradável e comida deliciosa.\n\n2. **Box 32**  \n   Localização: Mercado Público de Florianópolis  \n   Destaque: Comida típica catarinense e frutos do mar. É um dos pontos mais procurados para experimentar pratos locais em um ambiente descontraído.\n\n3. **Restaurante Rancho Açoriano**  \n   Localização: Lagoa da Conceição  \n   Destaque: Pratos típicos da culinária açoriana e frutos do mar, com um ambiente rústico e acolhedor.\n\n4. **Bar do Deca**  \n   Localização: Lagoa da Conceição  \n   Destaque: Petiscos e pratos com frutos do mar, ótimo para um happy hour com os locais.\n\n5. **Ponta das Caranhas**  \n   Localização: Ponta das Caranhas  \n   Destaque: Vista linda para o mar e pratos de frutos do mar frescos, muito frequentado por moradores.\n\n6. **Restaurante do Chico**  \n   Localização: Campeche  \n   Destaque: Ambiente simples e comida caseira, com destaque para peixes e frutos do mar.\n\nSe quiser opções mais específicas, como comida vegetariana ou internacional, posso ajudar também!</b></ul>
  <ul>]</ul>
}<br/>

In [12]:
set_debug(False)

chain2 = template_restaurante | llm | StrOutputParser() # A saída do llm é uma string, por isso, utilizamos o StrOutputParser
chain3 = template_cultural | llm |StrOutputParser() 

chain = chain1 | chain2 | chain3
                                 
resposta = chain.invoke({"interesse":"praias"}) 
print(resposta)

Claro! Além das opções gastronômicas que você mencionou, Florianópolis oferece diversas atividades e locais culturais que valem a pena conhecer. Aqui vão algumas sugestões para aproveitar a cultura local:

### Atividades culturais em Florianópolis

1. **Visita ao Mercado Público de Florianópolis**  
   - Localização: Centro  
   - O Mercado Público é um ponto histórico e cultural da cidade, com lojas, bares e restaurantes. Além de experimentar a culinária local, você pode encontrar artesanato, música ao vivo e eventos culturais.

2. **Museu Histórico de Santa Catarina (Palácio Cruz e Sousa)**  
   - Localização: Centro  
   - Um museu que conta a história da região e da cidade, instalado em um prédio histórico com arquitetura neoclássica.

3. **Projeto Tamar - Centro de Visitantes Florianópolis**  
   - Localização: Barra da Lagoa  
   - Um espaço dedicado à conservação das tartarugas marinhas, com exposições educativas e possibilidade de conhecer mais sobre a fauna marinha local.

4. 

#### <b>PASSO 4.1 - INVOCANDO A CADEIA GERAL, MAS PASSANDO SOMENTE A CIDADE PARA A CADEIA 3, NÃO TODA A STRING DE RESTAURANTES</b>

<font style="color:lightgreen">[llm/start]</font> [chain:RunnableSequence > chain:RunnableParallel<b><restaurantes,cidade,locais_culturais></b> > chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [<br/>
    <ul><ul>"Human: Sugira atividades e locais culturais em Florianópolis"</ul></ul>
  ]</ul>
}

In [13]:
chain = chain1 | {  # EXECUTANDO AS PARTES 2 E 3 AO MESMO TEMPO, AMBAS OBTENDO A CIDADE. A PARTE 2 AGORA ESTÁ RECEBENDO APENAS A CIDADE, E NÃO A STRING DA PARTE 2
                    "restaurantes" : chain2,
                    "locais_culturais": chain3 # AQUI TAMBÉM ESTAMOS PASSANDO A CIDADE PARA A PARTE 3
                 }

resposta = chain.invoke({"interesse":"praias"}) 
print(resposta)

{'restaurantes': 'Claro! Florianópolis tem uma cena gastronômica rica, com muitos restaurantes apreciados pelos moradores locais. Aqui estão algumas sugestões populares entre os locais:\n\n1. **Ostradamus** – Localizado na Lagoa da Conceição, é famoso por pratos com frutos do mar frescos, especialmente ostras e camarões. Ambiente descontraído e ótimo para quem quer experimentar a culinária local.\n\n2. **Bar do Arante** – Na Praia do Pântano do Sul, é um clássico para quem quer comer peixe fresco e frutos do mar em um ambiente simples e tradicional.\n\n3. **Restaurante Rancho Açoriano** – Também no Pântano do Sul, é conhecido por pratos típicos açorianos, como a sequência de camarão e o peixe na telha.\n\n4. **Box 32** – Localizado no Mercado Público de Florianópolis, é um lugar perfeito para experimentar pratos típicos da região, com destaque para o pastel de berbigão e a sequência de frutos do mar.\n\n5. **Canto dos Açores** – Restaurante tradicional que serve pratos típicos da culin

#### <b>PASSO 4.2 - INVOCANDO A CADEIA GERAL - PARA O MODELO FINAL</b>

<font style="color:lightgreen">[chain/start]</font> [chain:RunnableSequence > prompt:ChatPromptTemplate] Entering Prompt run with input:<br/>
{
  <ul><b>"restaurantes"</b>: "Claro! Florianópolis é conhecida pela sua excelente gastronomia, especialmente frutos do mar, mas também oferece uma variedade de opções para todos os gostos. Aqui estão alguns restaurantes populares entre os locais:\n\n1. **Ostradamus** (Ribeirão da Ilha)  \n   Famoso por suas ostras frescas e pratos de frutos do mar. Um clássico da ilha, muito frequentado por quem conhece bem a região.\n\n2. **Bar do Arante** (Ponta das Canas)  \n   Um bar e restaurante simples, mas muito tradicional, conhecido pelo ambiente descontraído e pratos típicos catarinenses, como a tainha na brasa.\n\n3. **Box 32** (Mercado Público)  \n   Localizado no Mercado Público de Florianópolis, é uma ótima opção para comer frutos do mar frescos e pratos locais em porções generosas.\n\n4. **Restaurante Rancho Açoriano** (Lagoa da Conceição)  \n   Serve pratos típicos da culinária açoriana, com destaque para o camarão e peixe preparados de forma tradicional.\n\n5. **Ponta das Caranhas** (Campeche)  \n   Restaurante com boa variedade de frutos do mar, ambiente agradável e frequentado por moradores da região.\n\n6. **Canto dos Açores** (Centro)  \n   Um restaurante que valoriza a culinária típica da ilha, com pratos caseiros e ingredientes locais.\n\nSe quiser sugestões para outros tipos de culinária ou ambientes, posso ajudar também!",<br/>
  <b>"locais_culturais"</b>: "Claro! Florianópolis é uma cidade rica em cultura, história e arte. Aqui estão algumas sugestões de atividades e locais culturais para você aproveitar:\n\n### Atividades Culturais\n1. **Visita ao Mercado Público de Florianópolis**  \n   Um ponto tradicional da cidade onde você pode experimentar a culinária local, comprar artesanato e conhecer a cultura açoriana.\n\n2. **Passeio pelo Centro Histórico**  \n   Explore a Praça XV de Novembro, a Catedral Metropolitana, o Palácio Cruz e Sousa e o Museu Histórico de Santa Catarina para entender a história da região.\n\n3. **Visita ao Museu Victor Meirelles**  \n   Localizado no centro, esse museu é dedicado ao pintor catarinense Victor Meirelles e abriga várias obras de arte e exposições temporárias.\n\n4. **Teatro Álvaro de Carvalho (TAC)**  \n   Assista a uma peça, concerto ou evento cultural nesse teatro histórico, um dos mais importantes da cidade.\n\n5. **Feiras de Artesanato**  \n   Visite feiras como a Feira de Artesanato da Lagoa da Conceição para encontrar produtos feitos por artesãos locais.\n\n6. **Passeios culturais em Santo Antônio de Lisboa**  \n   Um bairro com forte influência açoriana, onde você pode conhecer ateliês, galerias de arte e restaurantes típicos.\n\n### Locais Culturais\n- **Museu de Arte de Santa Catarina (MASC)**  \n  Museu com exposições de arte contemporânea e eventos culturais.\n\n- **Fortaleza de São José da Ponta Grossa**  \n  Uma fortaleza histórica com vista para a baía, que conta parte da história militar da ilha.\n\n- **Casa da Alfândega**  \n  Espaço cultural que abriga exposições, eventos e tem arquitetura histórica.\n\n- **Biblioteca Pública Municipal**  \n  Para quem gosta de leitura e eventos literários.\n\n### Eventos Culturais\n- **Festival de Dança de Florianópolis**  \n  Um dos maiores festivais de dança do Brasil, com apresentações de grupos locais e nacionais.\n\n- **Festa do Divino Espírito Santo**  \n  Tradicional festa religiosa com manifestações culturais e gastronômicas típicas.\n\nSe quiser, posso ajudar a montar um roteiro personalizado de acordo com seus interesses!",<br/>
  <b>"cidade"</b>: "Florianópolis"</ul>
}
<font style="color:orange">[llm/start]</font> [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:<br/>
{
  <ul>"prompts": [
    "<b>AI: Sugestão de viagem para a cidade: Florianópolis\nAI: Restaurantes que você não pode perder: </b>Claro! Florianópolis é conhecida por sua excelente gastronomia, especialmente pratos à base de frutos do mar, além de opções variadas que agradam tanto turistas quanto moradores locais. Aqui estão alguns restaurantes populares entre os locais:\n\n1. **Box 32**  \n   Localizado no Mercado Público, é um dos pontos mais tradicionais para quem quer provar frutos do mar frescos, como ostras, camarões e peixes. O ambiente é simples, mas a qualidade da comida é muito elogiada.\n\n2. **Ostradamus**  \n   Em Ribeirão da Ilha, uma região famosa pela produção de ostras, este restaurante é muito apreciado por locais e visitantes. Além das ostras frescas, o cardápio traz pratos típicos da culinária açoriana.\n\n3. **Bar do Boni**  \n   Um bar e restaurante tradicional em Florianópolis, conhecido por petiscos, frutos do mar e pratos caseiros. O ambiente é descontraído, ótimo para uma refeição informal.\n\n4. **Restaurante do Chico**  \n   Localizado na Lagoa da Conceição, é famoso por pratos com frutos do mar e peixes frescos, além de um ambiente acolhedor e vista agradável.\n\n5. **Ponta das Caranhas**  \n   Também na Lagoa da Conceição, esse restaurante é muito querido pelos moradores pela qualidade dos pratos e pelo ambiente rústico e aconchegante.\n\n6. **Bar do Arante**  \n   Situado na Praia do Matadeiro, é um clássico para quem busca frutos do mar frescos em um ambiente rústico e à beira-mar.\n\nSe você quiser sugestões para tipos específicos de cozinha ou regiões da cidade, posso ajudar também!\n<b>AI: Atividades e locais culturais recomendados: </b>Claro! Florianópolis é uma cidade rica em cultura, com diversas opções de atividades e locais para quem quer conhecer mais sobre a história, arte e tradições locais. Aqui vão algumas sugestões:\n\n### Atividades culturais em Florianópolis\n\n1. **Visita ao Centro Histórico**  \n   Explore o centro da cidade, onde estão construções históricas como a Catedral Metropolitana, o Mercado Público e a Praça XV de Novembro. É um ótimo lugar para entender a história da cidade e apreciar a arquitetura colonial.\n\n2. **Museu Histórico de Santa Catarina (Palácio Cruz e Sousa)**  \n   Localizado no centro, o museu oferece exposições sobre a história e cultura catarinense, além de ser um belo prédio histórico.\n\n3. **Visita ao Museu de Arte de Santa Catarina (MASC)**  \n   Para quem gosta de arte contemporânea, o MASC tem um acervo interessante e exposições temporárias. Fica no centro da cidade.\n\n4. **Passeio pela Lagoa da Conceição**  \n   Além da beleza natural, a Lagoa é um ponto cultural com feiras de artesanato, música ao vivo e eventos culturais.\n\n5. **Festival de Dança de Joinville (próximo a Florianópolis)**  \n   Se estiver na região na época do festival (geralmente em julho), vale a pena visitar para assistir a apresentações de dança de alto nível.\n\n6. **Feiras de Artesanato e Gastronomia**  \n   Feiras como a Feira do Largo da Alfândega (aos sábados) e a Feira de Artesanato da Lagoa da Conceição são ótimas para conhecer o artesanato local e provar comidas típicas.\n\n7. **Teatro Álvaro de Carvalho (TAC)**  \n   Um dos teatros mais tradicionais da cidade, com programação variada de peças, shows e eventos culturais.\n\n### Locais culturais para visitar\n\n- **Fortaleza de Santa Cruz de Anhatomirim**  \n  Um forte histórico localizado na Ilha de Anhatomirim, acessível por barco, que conta a história da defesa da região.\n\n- **Projeto Tamar - Florianópolis**  \n  Um centro de conservação das tartarugas marinhas que também oferece atividades educativas e culturais.\n\n- **Caminho dos Açores**  \n  Uma rota cultural que passa por comunidades tradicionais açorianas na Ilha, onde é possível conhecer a cultura, arquitetura e gastronomia típica.\n\n- **Casa da Alfândega**  \n  Espaço cultural e gastronômico no centro, com eventos e exposições.\n\n- **Biblioteca Pública Municipal**  \n  Para quem gosta de leitura e eventos literários, é um espaço importante na cidade.\n\nSe quiser, posso ajudar a montar um roteiro mais detalhado de acordo com seus interesses!\n<b>System:</b> Combine as informações anteriores em 2 parágrafos coerentes"</ul>
  <ul>]</ul>
}

In [14]:
from operator import itemgetter # IMPORTANDO A FUNÇÃO ITEMGETTER PARA PODER PEGAR O VALOR DE UMA CHAVE DO DICIONÁRIO

set_debug(True)

template_final = ChatPromptTemplate.from_messages( # COMO SE EU JÁ TIVESSE PASSADO POR UM CHAT, E ESTOU QUERENDO CONTINUAR A CONVERSA
    [
    ("ai", "Sugestão de viagem para a cidade: {cidade}"), # RESPOSTA DA IA
    ("ai", "Restaurantes que você não pode perder: {restaurantes}"), # RESPOSTA DA IA
    ("ai", "Atividades e locais culturais recomendados: {locais_culturais}"), # RESPOSTA DA IA
    ("system", "Combine as informações anteriores em 2 parágrafos coerentes") # System Instructions são usadas para guiar o modelo a produzir 
                                                                              # uma saída mais estruturada e coerente
    ]
)

chain1 = template_cidade | llm | parseador
chain2 = template_restaurante | llm | StrOutputParser()
chain3 = template_cultural | llm | StrOutputParser()
chain4 = template_final | llm | StrOutputParser()

chain = (chain1 |
          # EXECUTANDO AO MESMO TEMPO AS CADEIAS 2 E 3, AMBAS OBTENDO A CIDADE.
          {
            "restaurantes" : chain2,
            "locais_culturais": chain3,
            "cidade" : itemgetter("cidade"),# "cidade" : itemgetter("cidade") significa que, ao avançar na cadeia, 
                                            # o valor da chave "cidade" do resultado anterior será usado como entrada para as próximas etapas, 
                                            # automatizando o fluxo de dados entre as partes do pipeline. Já que a saída da chain1 é um dicionário com a chave "cidade".            
          }
          | chain4) # ALÉM DE restaurantes e locais culturais, O TEMPLATE PRECISA RECEBER TAMBÉM A cidade. POR ISSO O itemgetter("cidade")

resultado = chain.invoke({"interesse" : "praias" })
print('\nResposta Final:\n',resultado)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "interesse": "praias"
}
[chain/start] [chain:RunnableSequence > prompt:PromptTemplate] Entering Prompt run with input:
{
  "interesse": "praias"
}
[chain/end] [chain:RunnableSequence > prompt:PromptTemplate] s] Exiting Prompt run with output:
[outputs]
[llm/start] [chain:RunnableSequence > llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: Sugira uma cidade, dado o meu interesse por praias.\n                                    The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {\"properties\": {\"foo\": {\"title\": \"Foo\", \"description\": \"a list of strings\", \"type\": \"array\", \"items\": {\"type\": \"string\"}}}, \"required\": [\"foo\"]}\nthe object {\"foo\": [\"bar\", \"baz\"]} is a well-formatted instance of the schema. The object {\"properties\": {\"foo\": [\"bar\", \"baz\"]}} is not well-formatted.\n\nHere is

#### <b>OUTRO FORMA DE CÓDIGO</b><br/>
#### <b>ADICIONANDO INFORMAÇÕES AO DICIONÁRIO DE ENTRADA</b><br/>
O trecho selecionado constrói uma cadeia de processamento (pipeline) utilizando o LangChain, onde cada etapa transforma ou adiciona informações ao dicionário de entrada, de forma sequencial e automatizada.

Veja como funciona cada parte:

{"queixa": RunnablePassthrough()}
Aqui, o pipeline começa recebendo um dicionário com a chave "queixa". O RunnablePassthrough() simplesmente repassa o valor recebido, garantindo que a chave "queixa" esteja disponível para as próximas etapas.

| RunnablePassthrough.assign(resultado_analise=parte1)
Nesta etapa, o método assign executa o componente parte1 (que provavelmente analisa o texto da queixa) e adiciona o resultado ao dicionário sob a chave "resultado_analise". Assim, o dicionário agora contém tanto a queixa original quanto o resultado da análise.

| RunnablePassthrough.assign(sentimento=parte2)
Da mesma forma, esta etapa executa parte2 (que avalia o sentimento da análise anterior) e adiciona o resultado sob a chave "sentimento". O dicionário passa a ter as chaves "queixa", "resultado_analise" e "sentimento".

| parte3
Por fim, o pipeline executa parte3, que utiliza o campo "sentimento" para gerar uma resposta final, provavelmente formatando ou concluindo o processamento da queixa.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
import os
from dotenv import load_dotenv
from langchain_core.runnables import RunnablePassthrough
from langchain.globals import set_debug
set_debug(True)

load_dotenv()

llm = ChatOpenAI(
    model="gpt-4-0125-preview",
    temperature=0.5,
    api_key=os.getenv("OPENAI_KEY"))

parte1 = PromptTemplate.from_template("Analisar a queixa: {queixa}") | llm | StrOutputParser() # Runnable parte1
parte2 = PromptTemplate.from_template("Avaliar sentimento da queixa: {resultado_analise}") | llm | StrOutputParser() # Runnable parte2
parte3 = PromptTemplate.from_template("Formular resposta: {sentimento}") | llm | StrOutputParser() # Runnable parte 3

cadeia = ( # CONSTRÓI O PIPELINE QUE ENRIQUECE UM DICIONÁRIO DE ENTRADA
    {"queixa": RunnablePassthrough()} # DICIONÁRIO DE ENTRADA QUE SERÁ ENRIQUECIDO
                                      # RunnablePassthrough: utilizado para mapear e passar dados através 
                                      # da cadeia.
                                      # Garante que a chave "queixa" do dicionário de entrada, seja passada 
                                      # para a próxima etapa da cadeia.
    | RunnablePassthrough.assign(resultado_analise=parte1) # EXECUTA A PARTE 1 USANDO OS DADOS ATUAIS E SALVA
                                                           # A SAÍDA DA PARTE 1 NA CHAVE 
                                                           # resultado_analise DO DICIONÁRIO                                       
    | RunnablePassthrough.assign(sentimento=parte2) # RunnablePassthrough: utilizado para mapear e passar dados através da cadeia.
    | parte3
)

queixa_texto = """Hoje comprei um telefone novo, modelo X com 256 GB e flip. No entanto, o produto apresentou defeito na dobradiça e não permanece fechado. 
                    O suporte não me atende e estou super arrependido."""
                    
resultado = cadeia.invoke({"queixa": queixa_texto})

print(resultado)

[chain/start] [chain:RunnableSequence] Entering Chain run with input:
{
  "queixa": "Hoje comprei um telefone novo, modelo X com 256 GB e flip. No entanto, o produto apresentou defeito na dobradiça e não permanece fechado. \n                    O suporte não me atende e estou super arrependido."
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<queixa>] Entering Chain run with input:
{
  "queixa": "Hoje comprei um telefone novo, modelo X com 256 GB e flip. No entanto, o produto apresentou defeito na dobradiça e não permanece fechado. \n                    O suporte não me atende e estou super arrependido."
}
[chain/start] [chain:RunnableSequence > chain:RunnableParallel<queixa> > chain:RunnablePassthrough] Entering Chain run with input:
{
  "queixa": "Hoje comprei um telefone novo, modelo X com 256 GB e flip. No entanto, o produto apresentou defeito na dobradiça e não permanece fechado. \n                    O suporte não me atende e estou super arrependido."
}
[chain/en